<a href="https://colab.research.google.com/github/Rishabh-Creator-cyber/Capstone_Project/blob/main/Capstone_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sqlite3, random, csv
from datetime import date, timedelta

random.seed(42)

conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS category_targets;

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT NOT NULL,
    supplier TEXT NOT NULL,
    unit_price_inr INTEGER NOT NULL
);
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    signup_date TEXT NOT NULL,
    city TEXT NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    amount_inr INTEGER NOT NULL,
    payment_mode TEXT NOT NULL,
    status TEXT NOT NULL,
    rating INTEGER,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
CREATE TABLE category_targets (
    category TEXT PRIMARY KEY,
    target_revenue_inr INTEGER NOT NULL
);
""")

cities = ["Bengaluru", "Mumbai", "Hyderabad", "Pune"]

products_raw = [
    ("Banana 1kg", "Fruits & Vegetables", "FreshFarms Co", 50),
    ("Tomato 1kg", "Fruits & Vegetables", "FreshFarms Co", 40),
    ("Onion 1kg", "Fruits & Vegetables", "GreenValley Traders", 35),
    ("Apple 1kg", "Fruits & Vegetables", "GreenValley Traders", 180),
    ("Spinach Bunch", "Fruits & Vegetables", "FreshFarms Co", 25),
    ("Toned Milk 1L", "Dairy & Eggs", "DairyBest Ltd", 60),
    ("Paneer 200g", "Dairy & Eggs", "DairyBest Ltd", 90),
    ("Eggs (12pc)", "Dairy & Eggs", "CountryEggs Farms", 84),
    ("Curd 400g", "Dairy & Eggs", "DairyBest Ltd", 45),
    ("Butter 100g", "Dairy & Eggs", "CountryEggs Farms", 55),
    ("Potato Chips 90g", "Snacks & Beverages", "SnackHub India", 30),
    ("Cola 750ml", "Snacks & Beverages", "SnackHub India", 45),
    ("Biscuit Pack", "Snacks & Beverages", "BakeHouse Supplies", 35),
    ("Fruit Juice 1L", "Snacks & Beverages", "SnackHub India", 110),
    ("Namkeen 200g", "Snacks & Beverages", "BakeHouse Supplies", 60),
    ("Shampoo 340ml", "Personal Care", "CarePlus Distributors", 220),
    ("Toothpaste 150g", "Personal Care", "CarePlus Distributors", 95),
    ("Soap Bar 125g", "Personal Care", "CarePlus Distributors", 40),
    ("Hand Wash 250ml", "Personal Care", "CarePlus Distributors", 99),
    ("Face Wash 100g", "Personal Care", "CarePlus Distributors", 150),
    ("Dish Wash Bar", "Household Essentials", "HomeEssentials Traders", 20),
    ("Detergent 1kg", "Household Essentials", "HomeEssentials Traders", 130),
    ("Floor Cleaner 1L", "Household Essentials", "HomeEssentials Traders", 145),
    ("Toilet Cleaner 500ml", "Household Essentials", "HomeEssentials Traders", 89),
    ("Garbage Bags (30pc)", "Household Essentials", "HomeEssentials Traders", 75),
    ("Bread Loaf", "Bakery", "BakeHouse Supplies", 45),
    ("Croissant (2pc)", "Bakery", "BakeHouse Supplies", 70),
    ("Muffin Pack (4pc)", "Bakery", "BakeHouse Supplies", 120),
    ("Cake Slice", "Bakery", "BakeHouse Supplies", 85),
    ("Cookies 200g", "Bakery", "BakeHouse Supplies", 65),
    ("Premium Face Cream 50g", "Personal Care", "CarePlus Distributors", 450),
]
# Note: "Premium Face Cream 50g" (product_id 31) is deliberately excluded from
# popularity_weights/product_ids_weighted below, so it never receives an order —
# this gives the Task 4(b) LEFT JOIN a genuine zero-order row to preserve.
products = [(i, *p) for i, p in enumerate(products_raw, start=1)]
cur.executemany("INSERT INTO products VALUES (?,?,?,?,?)", products)

first_names = ["Aarav","Vivaan","Aditya","Vihaan","Arjun","Sai","Reyansh","Ayaan","Krishna","Ishaan",
               "Ananya","Diya","Saanvi","Aadhya","Kiara","Myra","Anika","Navya","Riya","Siya",
               "Rohan","Kabir","Dev","Yash","Aryan","Zara","Meera","Tara","Nisha","Priya",
               "Aman","Rahul","Karan","Varun","Nikhil","Pooja","Neha","Simran","Divya","Isha",
               "Rohit","Sanjay","Vikram","Manish","Deepak","Kavya","Shreya","Anjali","Pallavi","Sneha"]
customers = []
start_signup = date(2025, 1, 1)
for i, fname in enumerate(first_names, start=1):
    city = cities[i % len(cities)]
    signup = start_signup + timedelta(days=random.randint(0, 400))
    customers.append((i, fname, signup.isoformat(), city))
cur.executemany("INSERT INTO customers VALUES (?,?,?,?)", customers)

popularity_weights = [10,8,6,3,5, 9,7,6,8,4, 10,9,5,4,6, 3,5,9,4,3, 8,6,5,4,7, 6,4,3,5,4]
order_date_start = date(2026, 1, 1)
order_date_end = date(2026, 6, 30)
total_days = (order_date_end - order_date_start).days

TOTAL_ORDERS = 500
product_ids_weighted = []
for pid, w in zip(range(1, 31), popularity_weights):
    product_ids_weighted.extend([pid] * w)

payment_modes = ["UPI", "Credit Card", "Debit Card", "Cash on Delivery", "Wallet"]
product_lookup = {p[0]: p for p in products}
customer_lookup = {c[0]: c for c in customers}

orders = []
order_id = 1
for _ in range(TOTAL_ORDERS):
    cust_id = random.randint(1, 50)
    prod_id = random.choice(product_ids_weighted)
    unit_price = product_lookup[prod_id][4]
    quantity = random.randint(1, 5)
    amount = quantity * unit_price
    day_offset = random.randint(0, total_days)
    o_date = order_date_start + timedelta(days=day_offset)
    payment_mode = random.choice(payment_modes)
    roll = random.random()
    if roll < 0.85:
        status = "Delivered"
        rating = random.randint(1, 5)
    elif roll < 0.95:
        status = "Cancelled"
        rating = None
    else:
        status = "Pending"
        rating = None
    orders.append([order_id, cust_id, prod_id, o_date.isoformat(), quantity, amount, payment_mode, status, rating])
    order_id += 1
cur.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?,?,?,?)", [tuple(o) for o in orders])

category_targets = [
    ("Fruits & Vegetables", 12000),
    ("Dairy & Eggs", 16500),
    ("Snacks & Beverages", 13000),
    ("Personal Care", 15500),
    ("Household Essentials", 17000),
    ("Bakery", 12000),
]
cur.executemany("INSERT INTO category_targets VALUES (?,?)", category_targets)

conn.commit()

# --- Raw exports for Part 4 (Python/Pandas) — deliberately messy, do not "fix" here ---
raw_rows = []
for o in orders:
    order_id, cust_id, prod_id, o_date, qty, amt, pm, status, rating = o
    cust = customer_lookup[cust_id]
    raw_rows.append({
        "order_id": order_id, "order_date": o_date, "customer_name": cust[1],
        "city": cust[3], "category": product_lookup[prod_id][2], "product_id": prod_id,
        "quantity": qty, "amount_inr": amt, "payment_mode": pm, "status": status,
        "rating": rating if rating is not None else "",
    })

rng2 = random.Random(7)
casing_idx = rng2.sample(range(len(raw_rows)), 20)
for idx in casing_idx:
    r = raw_rows[idx]
    variant = rng2.choice(["upper", "lower", "space"])
    r["city"] = r["city"].upper() if variant == "upper" else (r["city"].lower() if variant == "lower" else " " + r["city"] + " ")
    variant2 = rng2.choice(["upper", "lower", "space"])
    r["category"] = r["category"].upper() if variant2 == "upper" else (r["category"].lower() if variant2 == "lower" else " " + r["category"] + " ")

null_idx = rng2.sample([i for i in range(len(raw_rows)) if i not in casing_idx], 10)
for idx in null_idx:
    raw_rows[idx]["amount_inr"] = ""

outlier_pool = [i for i in range(len(raw_rows)) if i not in casing_idx and i not in null_idx]
outlier_idx = rng2.sample(outlier_pool, 5)
for idx in outlier_idx:
    raw_rows[idx]["amount_inr"] = raw_rows[idx]["amount_inr"] * 40

dup_pool = [i for i in range(len(raw_rows)) if i not in casing_idx and i not in null_idx and i not in outlier_idx]
dup_idx = rng2.sample(dup_pool, 8)
all_rows = raw_rows + [dict(raw_rows[i]) for i in dup_idx]

fieldnames = ["order_id","order_date","customer_name","city","category","product_id",
              "quantity","amount_inr","payment_mode","status","rating"]
with open("orders_raw.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    w.writerows(all_rows)

with open("products.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["product_id", "product_name", "category", "supplier", "unit_price_inr"])
    w.writerows(products)

conn.close()
print("bigbasket_capstone.db, orders_raw.csv, products.csv created.")



bigbasket_capstone.db, orders_raw.csv, products.csv created.


In [ ]:
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM Products")
print(cur.fetchone())

conn.close()

(31,)


In [ ]:
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM Customers")
print(cur.fetchone())

conn.close()

(50,)


In [ ]:
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM Orders")
print(cur.fetchone())

conn.close()

(500,)


In [ ]:
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute("SELECT COUNT(*) FROM category_targets")
print(cur.fetchone())

conn.close()

(6,)


In [ ]:
conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute("SELECT count(*) FROM Orders GROUP BY status")

for row in cur.fetchall():
    print(row)

conn.close()

(42,)
(434,)
(24,)


In [ ]:
print('SELECT/WHERE')

conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute(" SELECT count(*) from Customers WHERE city = 'Mumbai' ")
print('Total Number of orders in Mumbai:', cur.fetchall())

conn.close()

SELECT/WHERE
Total Number of orders in Mumbai: [(13,)]


In [ ]:
print('DISTINCT')

conn = sqlite3.connect("bigbasket_capstone.db")
cur = conn.cursor()

cur.execute(" SELECT DISTINCT category from Products")
for row in cur.fetchall():
  print(row)

conn.close()

DISTINCT
('Fruits & Vegetables',)
('Dairy & Eggs',)
('Snacks & Beverages',)
('Personal Care',)
('Household Essentials',)
('Bakery',)


In [ ]:
print('ORDER BY + LIMIT')

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(''' SELECT amount_inr
                  FROM Orders
                  ORDER BY amount_inr desc
                  LIMIT 5 ''')
print(cur.fetchall())
conn.close()


ORDER BY + LIMIT
[(1100,), (900,), (750,), (725,), (720,)]


In [ ]:
print('ALIAS')

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(''' SELECT sum(amount_inr) AS TOTAL_REVENUE,
                  order_date AS DATE_OF_HIGHEST_REVENUE
                  FROM Orders
                  GROUP BY order_date
                  ORDER BY amount_inr desc
                  LIMIT 5 ''')
for row in cur.fetchall():
  print(row)

conn.close()

ALIAS
(1080, '2026-01-26')
(1180, '2026-03-28')
(660, '2026-03-08')
(1933, '2026-02-25')
(755, '2026-01-12')


In [ ]:
print('IN')

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute("SELECT DISTINCT customer_id , payment_mode FROM Orders WHERE payment_mode IN ('UPI','Cash on Delivery') LIMIT 20")

for row in cur.fetchall():
  print(row)

conn.close()


IN
(23, 'UPI')
(24, 'UPI')
(50, 'UPI')
(14, 'Cash on Delivery')
(9, 'UPI')
(10, 'Cash on Delivery')
(8, 'Cash on Delivery')
(1, 'UPI')
(29, 'UPI')
(36, 'UPI')
(15, 'UPI')
(37, 'Cash on Delivery')
(43, 'Cash on Delivery')
(44, 'Cash on Delivery')
(16, 'Cash on Delivery')
(18, 'Cash on Delivery')
(7, 'UPI')
(4, 'Cash on Delivery')
(19, 'Cash on Delivery')
(14, 'UPI')


In [ ]:
print("BETWEEN")

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(' SELECT order_id , amount_inr FROM Orders WHERE amount_inr BETWEEN 500 and 1000 LIMIT 20')
for row in cur.fetchall():
  print(row)

conn.close()

BETWEEN
(77, 650)
(79, 600)
(80, 520)
(121, 580)
(122, 520)
(131, 580)
(134, 520)
(139, 580)
(152, 720)
(155, 520)
(183, 540)
(193, 600)
(197, 600)
(222, 660)
(232, 750)
(234, 650)
(248, 520)
(257, 520)
(282, 520)
(288, 725)


In [ ]:
print("NOT BETWEEN")

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(' SELECT order_id , amount_inr FROM Orders WHERE amount_inr NOT BETWEEN 50 and 600 LIMIT 20')
for row in cur.fetchall():
  print(row)

conn.close()

NOT BETWEEN
(4, 30)
(20, 45)
(21, 20)
(26, 40)
(27, 40)
(43, 45)
(52, 25)
(60, 30)
(66, 45)
(73, 45)
(77, 650)
(101, 20)
(116, 30)
(117, 20)
(119, 25)
(126, 30)
(129, 35)
(130, 40)
(138, 45)
(143, 45)


In [ ]:
print('IS NULL')

conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(' SELECT status , Rating FROM Orders WHERE rating IS NULL LIMIT 10')

for row in cur.fetchall():
  print(row)

conn.close()

IS NULL
('Pending', None)
('Cancelled', None)
('Pending', None)
('Cancelled', None)
('Cancelled', None)
('Cancelled', None)
('Cancelled', None)
('Cancelled', None)
('Pending', None)
('Cancelled', None)


02_aggeregation_joins.sql


In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute('''SELECT category ,
                  count(order_id) AS TOTAL_ORDERS,
                  sum(amount_inr) AS TOTAL_REVENUE ,
                  AVG(amount_inr) AS AVG_REVENUE
                 FROM orders as o INNER JOIN Products as p on o.product_id = p.product_id
                 WHERE o.status = 'Delivered'
                 GROUP BY category
                 HAVING TOTAL_REVENUE > 10000 ''')
for row in cur.fetchall():
  print(row)

conn.close()

('Bakery', 67, 15410, 230.0)
('Dairy & Eggs', 66, 14090, 213.4848484848485)
('Household Essentials', 79, 21715, 274.873417721519)
('Personal Care', 66, 16382, 248.21212121212122)
('Snacks & Beverages', 83, 10895, 131.26506024096386)


In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute('''SELECT p.product_name ,
                  count(o.order_id) AS TOTAL_ORDERS
                 FROM products as p LEFT JOIN orders as o on p.product_id = o.product_id
                 GROUP BY p.product_name
''')
for row in cur.fetchall():
  print(row)

conn.close()

('Apple 1kg', 6)
('Banana 1kg', 14)
('Biscuit Pack', 15)
('Bread Loaf', 19)
('Butter 100g', 12)
('Cake Slice', 18)
('Cola 750ml', 32)
('Cookies 200g', 15)
('Croissant (2pc)', 14)
('Curd 400g', 17)
('Detergent 1kg', 24)
('Dish Wash Bar', 21)
('Eggs (12pc)', 8)
('Face Wash 100g', 7)
('Floor Cleaner 1L', 14)
('Fruit Juice 1L', 7)
('Garbage Bags (30pc)', 22)
('Hand Wash 250ml', 12)
('Muffin Pack (4pc)', 15)
('Namkeen 200g', 14)
('Onion 1kg', 19)
('Paneer 200g', 15)
('Potato Chips 90g', 27)
('Premium Face Cream 50g', 0)
('Shampoo 340ml', 3)
('Soap Bar 125g', 32)
('Spinach Bunch', 16)
('Toilet Cleaner 500ml', 15)
('Tomato 1kg', 25)
('Toned Milk 1L', 26)
('Toothpaste 150g', 16)


In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute('''SELECT product_name ,
                  amount_inr,
                  CASE
                        WHEN amount_inr >= 3000 then 'High'
                        WHEN amount_inr >= 1000 then 'Medium'
                        ELSE 'Low'
                 END
                 FROM orders as o INNER JOIN Products as p on o.product_id = p.product_id
                 WHERE o.status = 'Delivered'
                 GROUP BY product_name
                 ''')
for row in cur.fetchall():
  print(row)

conn.close()

('Apple 1kg', 180, 'Low')
('Banana 1kg', 200, 'Low')
('Biscuit Pack', 175, 'Low')
('Bread Loaf', 225, 'Low')
('Butter 100g', 165, 'Low')
('Cake Slice', 85, 'Low')
('Cola 750ml', 45, 'Low')
('Cookies 200g', 65, 'Low')
('Croissant (2pc)', 140, 'Low')
('Curd 400g', 225, 'Low')
('Detergent 1kg', 520, 'Low')
('Dish Wash Bar', 20, 'Low')
('Eggs (12pc)', 168, 'Low')
('Face Wash 100g', 300, 'Low')
('Floor Cleaner 1L', 290, 'Low')
('Fruit Juice 1L', 220, 'Low')
('Garbage Bags (30pc)', 150, 'Low')
('Hand Wash 250ml', 396, 'Low')
('Muffin Pack (4pc)', 480, 'Low')
('Namkeen 200g', 60, 'Low')
('Onion 1kg', 175, 'Low')
('Paneer 200g', 450, 'Low')
('Potato Chips 90g', 30, 'Low')
('Shampoo 340ml', 1100, 'Medium')
('Soap Bar 125g', 120, 'Low')
('Spinach Bunch', 50, 'Low')
('Toilet Cleaner 500ml', 267, 'Low')
('Tomato 1kg', 80, 'Low')
('Toned Milk 1L', 120, 'Low')
('Toothpaste 150g', 475, 'Low')


In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute('''SELECT category,
                  strftime('%Y-%m',order_date) AS month,
                  count(order_id) AS ORDER_COUNT,
                  SUM(amount_inr) AS TOTAL_REVENUE,
                  AVG(amount_inr) AS AVG_REVENUE
                 FROM orders as o INNER JOIN Products as p on o.product_id = p.product_id
                 WHERE o.status = 'Delivered'
                 GROUP BY category , month
                 ORDER BY category , month
                 ''')
for row in cur.fetchall():
  print(row)

conn.close()

('Bakery', '2026-01', 6, 735, 122.5)
('Bakery', '2026-02', 8, 2440, 305.0)
('Bakery', '2026-03', 11, 3090, 280.90909090909093)
('Bakery', '2026-04', 13, 3115, 239.6153846153846)
('Bakery', '2026-05', 15, 3695, 246.33333333333334)
('Bakery', '2026-06', 14, 2335, 166.78571428571428)
('Dairy & Eggs', '2026-01', 9, 2035, 226.11111111111111)
('Dairy & Eggs', '2026-02', 8, 1935, 241.875)
('Dairy & Eggs', '2026-03', 15, 3163, 210.86666666666667)
('Dairy & Eggs', '2026-04', 9, 1650, 183.33333333333334)
('Dairy & Eggs', '2026-05', 14, 3479, 248.5)
('Dairy & Eggs', '2026-06', 11, 1828, 166.1818181818182)
('Fruits & Vegetables', '2026-01', 12, 1290, 107.5)
('Fruits & Vegetables', '2026-02', 13, 2090, 160.76923076923077)
('Fruits & Vegetables', '2026-03', 13, 2080, 160.0)
('Fruits & Vegetables', '2026-04', 14, 1400, 100.0)
('Fruits & Vegetables', '2026-05', 12, 1870, 155.83333333333334)
('Fruits & Vegetables', '2026-06', 9, 1060, 117.77777777777777)
('Household Essentials', '2026-01', 10, 2099, 20

In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute(""" SELECT c.category,
                (target_revenue_inr - total_revenue) as Variance,
                ((total_revenue - target_revenue_inr) * 100)/ target_revenue_inr AS percentage_variance,
                CASE
                      WHEN sum(total_revenue) >= target_revenue_inr
                          THEN 'Above Target'
                      WHEN ((total_revenue - target_revenue_inr) * 100.0) / target_revenue_inr <= 15.0
                          THEN 'Below Target - Watch'
                      ELSE 'Below Target - Critical'
                      END AS status_tag
                from category_targets c inner JOIN
                    (SELECT category,
                     strftime('%Y-%m',order_date) AS month,
                     count(order_id) AS ORDER_COUNT,
                     SUM(amount_inr) AS TOTAL_REVENUE,
                     AVG(amount_inr) AS AVG_REVENUE
                     FROM orders as o INNER JOIN Products as p on o.product_id = p.product_id
                     WHERE o.status = 'Delivered'
                     GROUP BY category , month
                     ORDER BY category , month) r on c.category = r.category
                 GROUP BY c.category """)
for row in cur.fetchall():
  print(row)

conn.close()

('Bakery', 11265, -93, 'Above Target')
('Dairy & Eggs', 14465, -87, 'Below Target - Watch')
('Fruits & Vegetables', 10710, -89, 'Below Target - Watch')
('Household Essentials', 14901, -87, 'Above Target')
('Personal Care', 11892, -76, 'Above Target')
('Snacks & Beverages', 11710, -90, 'Below Target - Watch')


In [ ]:
import sqlite3
import csv

conn = sqlite3.connect("bigbasket_capstone.db")
cursor = conn.cursor()
cursor.execute("""SELECT category,
                  strftime('%Y-%m',order_date) AS month,
                  count(order_id) AS ORDER_COUNT,
                  SUM(amount_inr) AS TOTAL_REVENUE,
                  AVG(amount_inr) AS AVG_REVENUE
                 FROM orders as o INNER JOIN Products as p on o.product_id = p.product_id
                 WHERE o.status = 'Delivered'
                 GROUP BY category , month
                 ORDER BY category , month""")

with open("monthly_category_revenue.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([col[0] for col in cursor.description])
    writer.writerows(cursor.fetchall())

conn.close()

In [ ]:
conn = sqlite3.connect('bigbasket_capstone.db')
cur = conn.cursor()

cur.execute("""
    SELECT
        c.category,
        c.target_revenue_inr,
        SUM(r.TOTAL_REVENUE) AS total_revenue,
        (c.target_revenue_inr - SUM(r.TOTAL_REVENUE)) AS Variance,
        ((SUM(r.TOTAL_REVENUE) - c.target_revenue_inr) * 100.0) / c.target_revenue_inr AS percentage_variance,
        CASE
            WHEN SUM(r.TOTAL_REVENUE) >= c.target_revenue_inr
                THEN 'Above Target'
            WHEN ((c.target_revenue_inr - SUM(r.TOTAL_REVENUE)) * 100.0) / c.target_revenue_inr <= 15.0
                THEN 'Below Target - Watch'
            ELSE 'Below Target - Critical'
        END AS status_tag
    FROM category_targets c
    INNER JOIN (
        SELECT
            p.category,
            strftime('%Y-%m', o.order_date) AS month,
            COUNT(o.order_id) AS ORDER_COUNT,
            SUM(o.amount_inr) AS TOTAL_REVENUE,
            AVG(o.amount_inr) AS AVG_REVENUE
        FROM orders AS o
        INNER JOIN Products AS p ON o.product_id = p.product_id
        WHERE o.status = 'Delivered'
        GROUP BY p.category, month
    ) r ON c.category = r.category
    GROUP BY c.category, c.target_revenue_inr
""")

for row in cur.fetchall():
    print(row)

conn.close()

('Bakery', 12000, 15410, -3410, 28.416666666666668, 'Above Target')
('Dairy & Eggs', 16500, 14090, 2410, -14.606060606060606, 'Below Target - Watch')
('Fruits & Vegetables', 12000, 9790, 2210, -18.416666666666668, 'Below Target - Critical')
('Household Essentials', 17000, 21715, -4715, 27.735294117647058, 'Above Target')
('Personal Care', 15500, 16382, -882, 5.690322580645161, 'Above Target')
('Snacks & Beverages', 13000, 10895, 2105, -16.192307692307693, 'Below Target - Critical')
